<a href="https://colab.research.google.com/github/hsmu-jeongeun/machine-learning-practice/blob/main/250416_cross_validation_hyperparameter_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 교차 검증과 그리드 서치

## 검증 데이터셋

In [1]:
import pandas as pd

wine = pd.read_csv('https://bit.ly/wine-date')

### 문제 1 : wine 데이터 확인

In [3]:
# wine 처음 5개 행 데이터 확인
wine.head(5)

,alcohol,sugar,pH,class
0,9.4,1.9,3.51,0.0
1,9.8,2.6,3.20,0.0
2,9.8,2.3,3.26,0.0
3,9.8,1.9,3.16,0.0
4,9.4,1.9,3.51,0.0


In [5]:
# wine 전체 행의 개수 확인
print(wine.shape[0])

6497


In [6]:
# wine 데이터 통계값 확인 (각 특성별 평균, 표준편차, 최소값, 최대값 등)
wine.describe()

,alcohol,sugar,pH,class
count,6497.000000,6497.000000,6497.000000,6497.000000
mean,10.491801,5.443235,3.218501,0.753886
std,1.192712,4.757804,0.160787,0.430779
min,8.000000,0.600000,2.720000,0.000000
25%,9.500000,1.800000,3.110000,1.000000
50%,10.300000,3.000000,3.210000,1.000000
75%,11.300000,8.100000,3.320000,1.000000
max,14.900000,65.800000,4.010000,1.000000


In [7]:
# 화이트 와인, 레드 와인 데이터 개수 확인
wine['class'].value_counts()

class
1.0    4898
0.0    1599
Name: count, dtype: int64

### 데이터셋 분류

In [8]:
data = wine[['alcohol', 'sugar', 'pH']].to_numpy()
target = wine['class'].to_numpy()

In [9]:
from sklearn.model_selection import train_test_split

train_input, test_input, train_target, test_target = train_test_split(
    data, target, test_size=0.2, random_state=42)

In [10]:
sub_input, val_input, sub_target, val_target = train_test_split(
    train_input, train_target, test_size=0.2, random_state=42)

In [11]:
print(sub_input.shape, val_input.shape)

(4157, 3) (1040, 3)


In [12]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=42)
dt.fit(sub_input, sub_target)

print(dt.score(sub_input, sub_target))
print(dt.score(val_input, val_target))

0.9971133028626413
0.864423076923077


## 교차 검증

In [13]:
from sklearn.model_selection import cross_validate

scores = cross_validate(dt, train_input, train_target)
print(scores)

{'fit_time': array([0.00864482, 0.00826073, 0.00621104, 0.00572705, 0.0050559 ]), 'score_time': array([0.00075531, 0.00096512, 0.00049996, 0.00043893, 0.00039601]), 'test_score': array([0.87019231, 0.84615385, 0.87680462, 0.84889317, 0.83541867])}


In [14]:
import numpy as np

print(np.mean(scores['test_score']))

0.8554925223957948


In [15]:
from sklearn.model_selection import StratifiedKFold

scores = cross_validate(dt, train_input, train_target, cv=StratifiedKFold())
print(np.mean(scores['test_score']))

0.8554925223957948


In [16]:
splitter = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_validate(dt, train_input, train_target, cv=splitter)
print(np.mean(scores['test_score']))

0.8581873425226026


## 하이퍼파라미터 튜닝

In [17]:
from sklearn.model_selection import GridSearchCV

params = {'min_impurity_decrease': [0.0001, 0.0002, 0.0003, 0.0004, 0.0005]}

In [18]:
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params, n_jobs=-1)

In [19]:
gs.fit(train_input, train_target)

GridSearchCV(estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'min_impurity_decrease': [0.0001, 0.0002, 0.0003,
                                                   0.0004, 0.0005]})

In [20]:
dt = gs.best_estimator_
print(dt.score(train_input, train_target))

0.9615162593804117


In [21]:
print(gs.best_params_)

{'min_impurity_decrease': 0.0001}


In [22]:
print(gs.cv_results_['mean_test_score'])

[0.86800067 0.86453617 0.86492226 0.86780891 0.86761605]


In [23]:
best_index = np.argmax(gs.cv_results_['mean_test_score'])
print(gs.cv_results_['params'][best_index])

{'min_impurity_decrease': 0.0001}


In [24]:
params = {'min_impurity_decrease': np.arange(0.0001, 0.001, 0.0001),
          'max_depth': range(5, 20, 1),
          'min_samples_split': range(2, 100, 10)
          }

In [25]:
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params, n_jobs=-1)
gs.fit(train_input, train_target)

/opt/anaconda3/lib/python3.12/site-packages/numpy/ma/core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


GridSearchCV(estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': range(5, 20),
                         'min_impurity_decrease': array([0.0001, 0.0002, 0.0003, 0.0004, 0.0005, 0.0006, 0.0007, 0.0008,
       0.0009]),
                         'min_samples_split': range(2, 100, 10)})

In [26]:
print(gs.best_params_)

{'max_depth': 14, 'min_impurity_decrease': 0.0004, 'min_samples_split': 12}


In [27]:
print(np.max(gs.cv_results_['mean_test_score']))

0.8683865773302731


In [28]:
# 교차검증 수행 시간 프린트
gs.cv_results_['mean_fit_time']

array([0.00487976, 0.00377574, 0.00348473, ..., 0.00269361, 0.00259957,
       0.00269327])

### 랜덤 서치

In [29]:
from scipy.stats import uniform, randint

In [30]:
# 균등 분포 샘플링
rgen = randint(0, 10)
rgen.rvs(10)

array([8, 2, 5, 5, 3, 0, 2, 7, 7, 9])

In [31]:
np.unique(rgen.rvs(1000), return_counts=True) # 빈도도 함께 출력

(array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]),
 array([108, 120, 100,  89,  90, 103,  94,  90,  98, 108]))

In [32]:
ugen = uniform(0, 1)
ugen.rvs(10)

array([0.36602513, 0.73288811, 0.04554379, 0.47760481, 0.01824823,
       0.94657859, 0.72503539, 0.9538429 , 0.14393118, 0.96485144])

In [33]:
params = {'min_impurity_decrease': uniform(0.0001, 0.001),
          'max_depth': randint(20, 50),
          'min_samples_split': randint(2, 25),
          'min_samples_leaf': randint(1, 25),
          }

In [34]:
from sklearn.model_selection import RandomizedSearchCV

rs = RandomizedSearchCV(DecisionTreeClassifier(random_state=42), params,
                        n_iter=100, n_jobs=-1, random_state=42)
rs.fit(train_input, train_target)

RandomizedSearchCV(estimator=DecisionTreeClassifier(random_state=42),
                   n_iter=100, n_jobs=-1,
                   param_distributions={'max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x14386df40>,
                                        'min_impurity_decrease': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x14386f7a0>,
                                        'min_samples_leaf': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x14386e2d0>,
                                        'min_samples_split': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x14386e660>},
                   random_state=42)

In [35]:
print(rs.best_params_)

{'max_depth': 39, 'min_impurity_decrease': 0.00034102546602601173, 'min_samples_leaf': 7, 'min_samples_split': 13}


In [36]:
print(np.max(rs.cv_results_['mean_test_score']))

0.8695428296438884


In [37]:
dt = rs.best_estimator_

print(dt.score(test_input, test_target))

0.86


In [38]:
rs.cv_results_['mean_fit_time']

array([0.00314679, 0.00267601, 0.00288391, 0.00301065, 0.00253363,
       0.00264311, 0.00262942, 0.00257158, 0.00275621, 0.00273328,
       0.00278368, 0.002531  , 0.00283237, 0.00301471, 0.00264888,
       0.00297117, 0.00275474, 0.00313129, 0.00280299, 0.00326366,
       0.00316796, 0.00263124, 0.00273194, 0.0028626 , 0.00262537,
       0.0041925 , 0.00296521, 0.00274429, 0.00337553, 0.00263743,
       0.00412755, 0.00261526, 0.00256286, 0.0030654 , 0.00467854,
       0.00290513, 0.00365372, 0.00361958, 0.00285521, 0.00315065,
       0.0028296 , 0.00349555, 0.0039113 , 0.00339365, 0.00352964,
       0.00303531, 0.00321546, 0.00408688, 0.00403728, 0.00309334,
       0.00288534, 0.00298076, 0.0038878 , 0.00395832, 0.0027575 ,
       0.00320964, 0.00310202, 0.00389776, 0.00357871, 0.00279183,
       0.0036859 , 0.0026804 , 0.00266895, 0.00275435, 0.0028625 ,
       0.00428252, 0.00351472, 0.00298367, 0.00365629, 0.0026897 ,
       0.00392075, 0.00305576, 0.00313225, 0.00451951, 0.00311

In [39]:
print(np.mean(rs.cv_results_['mean_fit_time']))

0.003147252559661865


### 결정트리 분할 옵션 변경

In [40]:
rs2 = RandomizedSearchCV(DecisionTreeClassifier(splitter='random', random_state=42), params,
                        n_iter=100, n_jobs=-1, random_state=42)
rs2.fit(train_input, train_target)

RandomizedSearchCV(estimator=DecisionTreeClassifier(random_state=42,
                                                    splitter='random'),
                   n_iter=100, n_jobs=-1,
                   param_distributions={'max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x14386df40>,
                                        'min_impurity_decrease': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x14386f7a0>,
                                        'min_samples_leaf': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x14386e2d0>,
                                        'min_samples_split': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x14386e660>},
                   random_state=42)

In [41]:
print(rs2.best_params_)
print(np.max(rs2.cv_results_['mean_test_score']))

dt = rs2.best_estimator_
print(dt.score(test_input, test_target))

{'max_depth': 43, 'min_impurity_decrease': 0.00011407982271508446, 'min_samples_leaf': 19, 'min_samples_split': 18}
0.8458726956392981
0.786923076923077


In [42]:
rs2.cv_results_['mean_fit_time']

array([0.00145073, 0.0013814 , 0.00132093, 0.00119257, 0.0010798 ,
       0.00118647, 0.00119934, 0.00111442, 0.00105758, 0.00115995,
       0.00102701, 0.00096478, 0.00104938, 0.00100255, 0.00101032,
       0.00127101, 0.00100636, 0.0011157 , 0.00132031, 0.00123544,
       0.00128431, 0.00097051, 0.00099788, 0.00107327, 0.00090389,
       0.00107965, 0.0009932 , 0.00098987, 0.00095105, 0.00094419,
       0.00091457, 0.0010046 , 0.00084209, 0.00107837, 0.00105896,
       0.00106602, 0.00102963, 0.00098333, 0.00114818, 0.00145206,
       0.00118856, 0.00161524, 0.00099149, 0.00113726, 0.00159216,
       0.00101857, 0.00110264, 0.00104771, 0.00101867, 0.00145955,
       0.00101089, 0.00091801, 0.00122876, 0.00142546, 0.00097513,
       0.00189362, 0.00127883, 0.00108695, 0.00092916, 0.00088696,
       0.00102859, 0.00094571, 0.00089784, 0.0008934 , 0.00092287,
       0.00083671, 0.00081668, 0.0016428 , 0.00116735, 0.00088139,
       0.00116787, 0.00088506, 0.00119081, 0.00113978, 0.00098

In [74]:
print(np.mean(rs2.cv_results_['mean_fit_time']))

0.00974303340911865


문제 2 : 위 코드가 기존 랜덤 서치 코드와 다른 점을 2가지 적어보세요.

1.splitter='random' 옵션
2. mean-test-score가 기존 랜덤서치가 더 높다.